In [0]:
dbutils.widgets.text("env", "dev")
env = dbutils.widgets.get("env")

catalog = f"{env}_lakehouse"


In [0]:
bronze_df = spark.table(f"{catalog}.bronze.stock_prices_raw")


In [0]:
bronze_success_df = bronze_df.filter(bronze_df.api_status == "SUCCESS")


In [0]:
from pyspark.sql.functions import col, from_json, explode, to_date
from pyspark.sql.types import MapType, StringType


In [0]:
time_series_schema = MapType(
    StringType(),
    MapType(StringType(), StringType())
)


In [0]:
from pyspark.sql.types import StructType, StructField, StringType, MapType


In [0]:
alpha_schema = StructType([
    StructField("Meta Data", MapType(StringType(), StringType()), True),
    StructField(
        "Time Series (Daily)",
        MapType(
            StringType(),               # date
            MapType(StringType(), StringType())  # metrics
        ),
        True
    )
])


In [0]:
from pyspark.sql.functions import from_json, col


In [0]:
bronze_parsed_df = bronze_success_df.withColumn(
    "json_data",
    from_json(col("raw_payload"), alpha_schema)
)


In [0]:
time_series_df = bronze_parsed_df.withColumn(
    "time_series",
    col("json_data")["Time Series (Daily)"]
)


In [0]:
from pyspark.sql.functions import explode


In [0]:
exploded_df = time_series_df.select(
    col("symbol"),
    explode(col("time_series")).alias("trade_date", "metrics"),
    col("ingestion_ts")
)


In [0]:
from pyspark.sql.functions import to_date


In [0]:
silver_df = exploded_df.select(
    col("symbol"),
    to_date(col("trade_date")).alias("trade_date"),
    col("metrics")["1. open"].cast("double").alias("open"),
    col("metrics")["2. high"].cast("double").alias("high"),
    col("metrics")["3. low"].cast("double").alias("low"),
    col("metrics")["4. close"].cast("double").alias("close"),
    col("metrics")["5. volume"].cast("bigint").alias("volume"),
    col("ingestion_ts"),
    col("symbol").alias("record_source")
)


In [0]:
silver_clean_df = silver_df.filter(
    (col("open") >= 0) &
    (col("high") >= 0) &
    (col("low") >= 0) &
    (col("close") >= 0) &
    (col("volume") >= 0) &
    col("trade_date").isNotNull()
)


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window_spec = (
    Window
    .partitionBy("symbol", "trade_date")
    .orderBy(col("ingestion_ts").desc())
)

silver_dedup_df = (
    silver_clean_df
    .withColumn("rn", row_number().over(window_spec))
    .filter(col("rn") == 1)
    .drop("rn")
)


In [0]:
dbutils.widgets.text("env", "dev")
env = dbutils.widgets.get("env")

spark.sql(f"USE CATALOG {env}_lakehouse")
spark.sql("USE SCHEMA silver")


In [0]:
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.stock_prices_cleaned")
